# 18 · t-аномалия: стабильность по разным сидам

Проверка «очищенная температура + аномалии отдельной фичей» (§14·Б) на
**разных седах** (42, 14, 88, 777, 2025) с гиперпараметрами из Optuna
(`models/optuna_fire_best.json`: iters=600, lr=0.0091, depth=10, l2=1.22).

Для каждого сида — пара прогонов на holdout: «без t» vs «с t-аномалией».
Сводим таблицу и смотрим, устойчив ли (положительный) эффект t-аномалии.


In [1]:
%matplotlib inline
import sys
import pathlib
import json

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool

import data_utils as du
import features as fe
import tte_pipeline as tte
from tte_experiments import (survival_metrics, pr_at_horizon,
                             predict_survival_discrete)

DATASET = du.find_dataset_dir()
HP = json.loads((DATASET.parent / "models" / "optuna_fire_best.json").read_text(encoding="utf-8"))
print("hp:", HP)


hp: {'iters': 600, 'lr': 0.009110184519215777, 'depth': 10, 'l2_leaf_reg': 1.2194310213973099}


In [2]:
# Панель fire + t-аномалия (агрегат уже собран в ноутбуке 17)
tobj = pd.read_parquet(DATASET / "temp_obj_bucket6h_anom.parquet")
tobj["об"] = tobj["об"].astype(str)
tobj["t_cnt_total"] = tobj["t_cnt_ok"].fillna(0) + tobj["t_cnt_bad"].fillna(0)
tobj["t_cnt_ok"] = tobj["t_cnt_ok"].fillna(0)
tobj["t_cnt_bad"] = tobj["t_cnt_bad"].fillna(0)
def status(r):
    if r["t_cnt_total"] <= 0:
        return np.nan
    if r["t_cnt_bad"] <= 0:
        return "норм"
    bm, bx = r["t_min_bad"], r["t_max_bad"]
    if not np.isnan(bx) and (np.isnan(bm) or abs(bx) >= abs(bm)):
        return f"out:{int(round(bx))}"
    return f"out:{int(round(bm))}"
tobj["t_status"] = tobj.apply(status, axis=1)
tobj["t_status_num"] = pd.factorize(tobj["t_status"], use_na_sentinel=True)[0]
tm = tobj[["об", "бакет", "t_cnt_ok", "t_mean_ok", "t_min_ok", "t_max_ok",
           "t_cnt_bad", "t_min_bad", "t_max_bad", "t_cnt_total",
           "t_status", "t_status_num"]].rename(columns={"об": "ид_объект"})

panel = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                    dtype={"ид_канала_данных": str, "ид_объект": str})
panel = panel.merge(tm, on=["ид_объект", "бакет"], how="left")
panel["t_есть_б"] = (panel["t_cnt_total"].fillna(0) > 0).astype(np.int8)
panel["t_has_bad"] = (panel["t_cnt_bad"].fillna(0) > 0).astype(np.int8)
panel["t_sum_ok_б"] = panel["t_mean_ok"].to_numpy() * panel["t_cnt_ok"].fillna(0).to_numpy()
panel = fe._add_bucket_windows(panel, ["t_sum_ok_б", "t_cnt_ok"],
                               windows=fe.BUCKET_WINDOWS, group_col="ид_объект")
for label in ("12ч", "24ч", "48ч", "7д", "30д"):
    panel[f"t_mean_ok_сум_{label}"] = (
        panel[f"t_sum_ok_б_сум_{label}"] / panel[f"t_cnt_ok_сум_{label}"].replace(0, np.nan))
print("панель:", panel.shape)


панель: (1204508, 171)


In [3]:
# Субъекты + колонки X (t-фичи отбираются по префиксу)
panel = tte.refit_z_train_only(panel)
panel = tte.add_series_context(panel, event_col="серьёзных")
subjects = tte.add_tte_labels(panel)
subjects["ид_объект_code"] = subjects["ид_объект"].astype("category").cat.codes
subjects["тип_датчика_code"] = subjects["тип_датчика"].astype("category").cat.codes
tr_all, va_all, ho_all = tte.split_subjects(subjects)
X_all = tte.subject_features(tr_all)
X_base = [c for c in X_all if not c.startswith("t_")]
T_ANOM = ["t_cnt_ok", "t_mean_ok", "t_min_ok", "t_max_ok", "t_cnt_bad",
          "t_min_bad", "t_max_bad", "t_есть_б", "t_has_bad"] +          [f"t_mean_ok_сум_{l}" for l in ("12ч", "24ч", "48ч", "7д", "30д")]
X_anom = X_base + [c for c in T_ANOM if c in X_all]
print("X_base:", len(X_base), "| t-аномалия фич:", len(X_anom) - len(X_base))


X_base: 137 | t-аномалия фич: 14


In [4]:
def run_model(Xc, hp, seed, tag):
    tr = tte.sample_subjects(tr_all, 15000, 10000, 10000, seed=seed)
    va = tte.sample_subjects(va_all, 2000, 2000, 2000, seed=seed + 1)
    ho = tte.sample_holdout(ho_all, 20000, seed=seed + 1000)
    X_tr, y_tr = tte.expand_person_time(tr, Xc, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(va, Xc, tte.HORIZON_BUCKETS)
    m = CatBoostClassifier(iterations=hp["iters"], learning_rate=hp["lr"],
                           depth=hp["depth"], l2_leaf_reg=hp["l2_leaf_reg"],
                           random_seed=seed, task_type="GPU",
                           loss_function="Logloss", eval_metric="Logloss",
                           early_stopping_rounds=300, verbose=0,
                           allow_writing_files=False)
    m.fit(Pool(X_tr, y_tr), eval_set=Pool(X_va, y_va))
    surv_tr = tte.make_surv_struct(tr["event_flag"], tr["obs_days"])
    surv_ho = tte.make_surv_struct(ho["event_flag"], ho["obs_days"])
    S_ho = predict_survival_discrete(m, ho, Xc)
    risk = 1 - S_ho[:, -1]
    met = survival_metrics(surv_tr, surv_ho, risk, S_ho, tte.EVAL_TIMES)
    y24 = ((ho["event_flag"] == 1) & (ho["obs_days"] <= 1.01)).to_numpy(int)
    p24 = 1 - S_ho[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24)
    return met["uno_c"], pr["pr_auc"]

SEEDS = [42, 14, 88, 777, 2025]
rows = []
for s in SEEDS:
    u_ref, p_ref = run_model(X_base, HP, s, "без t")
    u_anom, p_anom = run_model(X_anom, HP, s, "t-аномалия")
    rows.append({"seed": s, "uno_без_t": u_ref, "uno_t": u_anom,
                 "pr_без_t": p_ref, "pr_t": p_anom})
    print(f"seed={s}: без t {u_ref:.3f}/{p_ref:.3f} | t-аном "
          f"{u_anom:.3f}/{p_anom:.3f}", flush=True)

res = pd.DataFrame(rows)
res["d_uno"] = res["uno_t"] - res["uno_без_t"]
res["d_pr"] = res["pr_t"] - res["pr_без_t"]
print()
print(res.round(4).to_string(index=False))
print()
print("mean: d_uno =", round(res["d_uno"].mean(), 4),
      "| d_pr =", round(res["d_pr"].mean(), 4))
print("улучшение PR-AUC в K сидах из", len(res),
      "=", int((res["d_pr"] > 0).sum()))


seed=42: без t 0.814/0.358 | t-аном 0.774/0.357


seed=14: без t 0.836/0.362 | t-аном 0.810/0.360


seed=88: без t 0.809/0.395 | t-аном 0.817/0.378


seed=777: без t 0.833/0.420 | t-аном 0.774/0.310


seed=2025: без t 0.805/0.374 | t-аном 0.793/0.358



 seed  uno_без_t  uno_t  pr_без_t   pr_t   d_uno    d_pr
   42     0.8139 0.7743    0.3582 0.3566 -0.0396 -0.0016
   14     0.8362 0.8098    0.3624 0.3602 -0.0264 -0.0022
   88     0.8086 0.8171    0.3951 0.3779  0.0085 -0.0172
  777     0.8331 0.7737    0.4202 0.3096 -0.0595 -0.1106
 2025     0.8047 0.7932    0.3741 0.3575 -0.0115 -0.0166

mean: d_uno = -0.0257 | d_pr = -0.0296
улучшение PR-AUC в K сидах из 5 = 0


### Интерпретация

- Смотрим средние `d_uno` и `d_pr` и сколько сидов из 5 дали положительный
  сдвиг. Если направление устойчиво (≥4/5, средняя дельта > 0) — фичу
  принимаем и переносим в дефолтный пайплайн; если результат шумный — фиксируем
  как «в пределах шума» и в дефолты не включаем.
